# PPO Hyperparameter Tuning

Logan Wong

law3082

In [1]:
import os
import numpy as np
import pandas as pd
import random
import pickle
import time
import port_for
import subprocess

# Env
import gymnasium as gym
from gymnasium import spaces
import torch

# Algo
from stable_baselines3 import PPO
from stable_baselines3.common.vec_env import DummyVecEnv

In [2]:
os.environ["CUDA_VISIBLE_DEVICES"]="6"

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

cuda


In [3]:
seed = 316
random.seed(seed)
np.random.seed(seed)

In [4]:
def convert(seconds):
    seconds = seconds % (24 * 3600)
    hour = seconds // 3600
    seconds %= 3600
    minutes = seconds // 60
    seconds %= 60

    return "%d:%02d:%02d" % (hour, minutes, seconds)

# Load data

In [5]:
# train_sequences_idx is a Dictionary
# 6040 keys (User indices, NOT User IDs, 0 to 6039)
# Values: list of movie indices, the first 80% of each user's ratings (all ratings 1-5, not filtered for positivity)
with open('../data/train_sequences_idx_64.pkl', 'rb') as f:
    train_sequences_idx = pickle.load(f)

# For Evaluation
with open('../data/valid_sequences_idx.pkl', 'rb') as f:
    valid_sequences_idx = pickle.load(f)

with open('../data/idx2user.pkl', 'rb') as f:
    idx2user = pickle.load(f)
with open('../data/user2idx.pkl', 'rb') as f:
    user2idx = pickle.load(f)

# ratings_dict is a Dictionary
# Each key is a tuple (user index, movie index)
# Values are that user's rating for that movie
with open('../data/ratings_dict.pkl', 'rb') as f:
    ratings_dict = pickle.load(f)

# user_embeddings is a 6040 x 100 matrix
user_embeddings = np.load('../data/user_embeddings.npy')

# movie_embeddings is a 3706 x 100 matrix
movie_embeddings = np.load('../data/movie_embeddings.npy')

with open('../data/movie_index_to_title.pkl', 'rb') as f:
    movie_index_to_title = pickle.load(f)


# Environment

In [6]:
class TrainPPORecommendationEnv(gym.Env):
    def __init__(self, 
                 train_sequences_idx,    # dict {user_idx: list of movie indices}
                 ratings_dict,          # dict {(user_idx, movie_idx): rating}
                 user_embeddings,       # numpy array (num_users, k)
                 movie_embeddings,      # numpy array (num_movies, k)
                 n_history=5,           # length of positive history
                 T_max=10):              # max steps per episode
        super().__init__()
        self.user_sequences = train_sequences_idx
        self.ratings = ratings_dict
        self.user_embeddings = user_embeddings
        self.movie_embeddings = movie_embeddings
        self.n_history = n_history
        self.T_max = T_max

        # Embedding dimensions
        self.k = movie_embeddings.shape[1]  # 100

        # Observation space: state vector from DRR-ave (size = 3*k)
        self.observation_space = spaces.Box(low=-np.inf, high=np.inf, shape=(3*self.k,), dtype=np.float32)
        
        # Action space: continuous vector in [-1, 1]
        self.action_space = spaces.Box(low=-1.0, high=1.0, shape=(self.k,), dtype=np.float32)

        # Create state representation module: Implement DRR-ave using numpy
        # Learnable item weights (initialized to 1)
        self.item_weights = np.ones(n_history)

        # Internal variables
        self.current_user = None
        self.history = None
        self.candidate_items = None
        self.recommended_set = None
        self.step_count = 0

    
    def _get_state(self, user_idx, history_indices):
        """Compute state vector: [u, u⊗g, g] with weighted average of history."""
        # u: user embedding
        u = self.user_embeddings[user_idx]                     # (k,)
        # g: weighted average of history items
        # Apply softmax to item weights to make them sum to 1
        weights = np.exp(self.item_weights) / np.sum(np.exp(self.item_weights))
        hist_embs = self.movie_embeddings[history_indices]      # (n_history, k)
        weighted_avg = np.sum(hist_embs * weights[:, np.newaxis], axis=0)  # (k,)
        # u⊗g element-wise product
        u_g = u * weighted_avg
        # Concatenate: [u, u⊗g, g]
        state = np.concatenate([u, u_g, weighted_avg])  # (3*k,)
        return state.astype(np.float32)

    def reset(self, *, seed=None, options=None):
        """
            Start a new episode for a given user.
            seed is passed in when the PPO model is created: PPO(seed=seed)
            stable‑baselines3 with Gymnasium expects options, so set it as None (but you never use it)
        """
        # Set random seed if provided
        if seed is not None:
            np.random.seed(seed)
        # Randomly select a user
        self.current_user = np.random.choice(list(self.user_sequences.keys()))
        seq = self.user_sequences[self.current_user]
    
        # Get MOST RECENT history: first n_history items (NOT the most recent positively rated movies)
        if len(seq) < self.n_history + 1:
            # Fallback: use whatever we have (but ensure we have enough for step)
            self.history = seq[:self.n_history]
        else:
            self.history = seq[:self.n_history].copy()
        # Get candidate set
        self.candidate_items = set(seq[self.n_history:])

        # Set initial state
        self.recommended_set = set()
        self.step_count = 0    
        state = self._get_state(self.current_user, self.history)
        return state, {}

    def step(self, action):
        """
        action: numpy array (k,) continuous vector.
        """

        # 1. Check that candidates exist
        candidates = list(self.candidate_items - self.recommended_set)
        if not candidates:
            # If no items left, then episode ends
            next_state = self._get_state(self.current_user, self.history)
            return next_state, 0.0, True, False, {}

        # 2. Compute scores for every candidate items (dot product with action)
        # Get embeddings of candidates
        cand_emb = self.movie_embeddings[candidates]   # num_cand x k matrix
        scores = cand_emb @ action                     # (num_cand,); 1 score per candidate

        # 3. Select highest scoring item
        best_idx = np.argmax(scores)
        selected_item = candidates[best_idx]

        # 4. Get reward
        rating = self.ratings.get((self.current_user, selected_item), 0)
        reward = (rating - 3) / 2.0

        # 5. Update state history
        new_history = self.history.copy()
        if reward > 0:   # positive feedback
            new_history = new_history[1:] + [selected_item]  # drop oldest, add new
        # else: keep unchanged

        # 6. Update sets
        self.recommended_set.add(selected_item)
        self.step_count += 1

        # 7. Check if you're done or not
        terminated = (len(self.candidate_items - self.recommended_set) == 0)  # no more items
        truncated = (self.step_count >= self.T_max)                           # max steps reached

        # 8. Next state
        next_state = self._get_state(self.current_user, new_history)
        # Update internal history
        self.history = new_history

        # Optional info
        info = {
            'selected_item': selected_item,
            'rating': rating,
            'reward': reward,
            'step': self.step_count
        }

        # Return according to Gymnasium API: obs, reward, terminated, truncated, info
        return next_state, reward, terminated, truncated, info

In [7]:
class EvalPPORecommendationEnv(gym.Env):
    def __init__(self, 
                 train_sequences_idx,      # dict: user_idx -> [movie_idx, ...]
                 test_sequences_idx,       # dict: user_idx -> [movie_idx, ...]
                 ratings_dict,             # (user_idx, movie_idx) -> rating
                 user_embeddings, movie_embeddings,
                 evaluable_users,
                 n_history=5, T_max=10):
        super().__init__()
        self.train = train_sequences_idx
        self.test = test_sequences_idx
        self.ratings = ratings_dict
        self.user_embeddings = user_embeddings
        self.movie_embeddings = movie_embeddings
        self.evaluable_users = evaluable_users
        self.n_history = n_history
        self.T_max = T_max

        self.k = movie_embeddings.shape[1]
        self.observation_space = spaces.Box(low=-np.inf, high=np.inf, shape=(3*self.k,), dtype=np.float32)
        self.action_space = spaces.Box(low=-1.0, high=1.0, shape=(self.k,), dtype=np.float32)
        self.item_weights = np.ones(self.n_history)

        self.current_user = None
        self.history = None
        self.candidate_items = None
        self.recommended_set = None
        self.step_count = 0

    def _get_state(self, user_idx, history_indices):
        u = self.user_embeddings[user_idx]
        weights = np.exp(self.item_weights) / np.sum(np.exp(self.item_weights))
        hist_embs = self.movie_embeddings[history_indices]
        weighted_avg = np.sum(hist_embs * weights[:, np.newaxis], axis=0)
        u_g = u * weighted_avg
        return np.concatenate([u, u_g, weighted_avg]).astype(np.float32)

    def reset(self, *, seed=None, options=None):
        if seed is not None:
            np.random.seed(seed)

        # During evaluation, I'll always call reset(options={'user_idx': user_idx})
        # options lets me pass a specific user Index
        if options is not None and 'user_idx' in options:
            user_idx = options['user_idx']
        else:
            # NOTE: This should never happen b/c reset() has options passed in with user_idx
            # And the eval loop is iterating thru all user indices.
            user_idx = np.random.choice(self.evaluable_users)
        self.current_user = user_idx

        # Build positive history from training (last n_history items with rating >=4)
        pos_items = []
        for m in self.train[user_idx]:
            if self.ratings.get((user_idx, m), 0) >= 4:
                pos_items.append(m)
        self.history = pos_items[-self.n_history:].copy()

        # Set initial state
        self.candidate_items = set(self.test[user_idx])
        self.recommended_set = set()
        self.step_count = 0
        state = self._get_state(user_idx, self.history)
        return state, {}

    def step(self, action):
        candidates = list(self.candidate_items - self.recommended_set)
        if not candidates:
            next_state = self._get_state(self.current_user, self.history)
            return next_state, 0.0, True, False, {}

        cand_emb = self.movie_embeddings[candidates]
        scores = cand_emb @ action
        best_idx = np.argmax(scores)
        selected_item = candidates[best_idx]

        rating = self.ratings.get((self.current_user, selected_item), 0)
        reward = (rating - 3) / 2.0

        new_history = self.history.copy()
        if reward > 0:
            new_history = new_history[1:] + [selected_item]

        self.recommended_set.add(selected_item)
        self.step_count += 1

        terminated = (len(self.candidate_items - self.recommended_set) == 0)
        truncated = (self.step_count >= self.T_max)

        next_state = self._get_state(self.current_user, new_history)
        self.history = new_history

        info = {'selected_item': selected_item, 'rating': rating, 'reward': reward, 'step': self.step_count}
        return next_state, reward, terminated, truncated, info

# Evaluation of Validation Set

In [8]:
def get_ground_truth(env, user_idx):
    """
        Returns a set of movie INDICES that the user rated positively (4 or 5)
    """
    truth = set()
    for movie_idx in env.test[user_idx]:
        if env.ratings.get((user_idx, movie_idx), 0) >= 4:
            truth.add(movie_idx)
    return truth

In [9]:
def evaluate_ppo(model, env, evaluable_users, top_k_list=[5, 10], max_steps=10, rating_threshold=4):
    metrics = {k: {'precision': [], 'ndcg': []} for k in top_k_list}

    for user_index in evaluable_users:
        user_id = idx2user[user_index]
        obs, _ = env.reset(options={'user_idx': user_index})
        recommended_items = []
        step = 0
        done = False
    
        ground_truth = get_ground_truth(env, user_index)
    
        while not done and step < max_steps:
            action, _ = model.predict(obs, deterministic=True)
            obs, reward, terminated, truncated, info = env.step(action)
            done = terminated or truncated
    
            if 'selected_item' in info:
                selected_item = info['selected_item']
                recommended_items.append(selected_item)
            step += 1
    
        # Compute metrics for this user
        for k in top_k_list:
            top_k = recommended_items[:k]
            hits = sum(1 for m in top_k if m in ground_truth)
            precision = hits / k
            dcg = sum(1 / np.log2(i+2) for i, m in enumerate(top_k) if m in ground_truth)
            idcg = sum(1 / np.log2(i+2) for i in range(min(len(ground_truth), k)))
            ndcg = dcg / idcg if idcg > 0 else 0.0
            metrics[k]['precision'].append(precision)
            metrics[k]['ndcg'].append(ndcg)
    
    # Aggregate results
    results = {}
    for k in top_k_list:
        results[f'Precision@{k}'] = np.mean(metrics[k]['precision'])
        results[f'NDCG@{k}'] = np.mean(metrics[k]['ndcg'])
    return results

In [10]:
# Create list of User Indices that you can legitimately evaluate
evaluable_users = []
for user_idx in train_sequences_idx.keys():
    # Count positive items in training
    pos_train = 0
    for movie_idx in train_sequences_idx[user_idx]:
        rating = ratings_dict.get((user_idx, movie_idx), 0)
        if rating >= 4:
            pos_train += 1

    # if # of positively rated movies in the train set < 5, skip the rest of the for loop
    if pos_train < 5:
        continue
    
    # Check test set has at least one item
    test_items = valid_sequences_idx.get(user_idx, [])
    if len(test_items) == 0:
        continue
    
    # Check test set has at least one positive item
    has_positive_test = False
    for movie_idx in test_items:
        rating = ratings_dict.get((user_idx, movie_idx), 0)
        if rating >= 4:
            has_positive_test = True
            break
    
    if not has_positive_test:
        continue
    
    evaluable_users.append(user_idx)

print(f"Number of evaluable users: {len(evaluable_users)}")

Number of evaluable users: 3319


# PPO

In [11]:
# Load tensorboard
port = port_for.get_port(None)
print(f"Starting TensorBoard on port: {port}")

# Kill existing tensorboard
subprocess.run(["pkill", "-f", "tensorboard"])

# Start tensorboard
subprocess.Popen(["tensorboard", "--logdir", "../results/", "--port", f"{port}", "--host", "0.0.0.0"])

# Wait a moment
time.sleep(3)

# Test connection
import requests
try:
    r = requests.get(f"http://localhost:{port}")
    print(f"ssh -L {port}:localhost:{port} -N law3082@narnia.gccis.rit.edu")
    print(f"TensorBoard is running! Status: {r.status_code}")
    print(f"http://localhost:{port}")
except:
    print("TensorBoard failed to start")

Starting TensorBoard on port: 11279


TensorFlow installation not found - running with reduced feature set.

NOTE: Using experimental fast data loading logic. To disable, pass
    "--load_fast=false" and report issues on GitHub. More details:
    https://github.com/tensorflow/tensorboard/issues/4784

TensorBoard 2.20.0 at http://0.0.0.0:11279/ (Press CTRL+C to quit)


ssh -L 11279:localhost:11279 -N law3082@narnia.gccis.rit.edu
TensorBoard is running! Status: 200
http://localhost:11279


In [12]:
hyperparams = {
    'learning_rate': [0.0001, 0.0003, 0.001],
    'n_steps': [1024, 2048, 4096],
    'batch_size': [64, 128, 256],
    'gamma': [0.95, 0.99, 0.995],
    'ent_coef': [0.0, 0.01, 0.05]
}

# hyperparams = {
#     'learning_rate': [0.001],
#     'n_steps': [1024],
#     'batch_size': [64],
#     'gamma': [0.99],
#     'ent_coef': [0.0, 0.01]
# }

model_name = "PPO_HP_Tuning"
best_ndcg = -float('inf')
best_config = None
config_number = 0
total_timesteps = 100_000

with open('../results/hp_tuning_log.txt', 'w') as log_file:
    # Write a header with timestamp
    log_file.write(f"Hyperparameter Tuning Log - Started at {time.ctime()}\n")
    log_file.write("="*80 + "\n")
        
    print("Training started")
    start_time = time.time()
    for lr in hyperparams['learning_rate']:
        for n_steps in hyperparams['n_steps']:
            for batch_size in hyperparams['batch_size']:
                for gamma in hyperparams['gamma']:
                    for ent_coef in hyperparams['ent_coef']:
                        print(f"\nCONFIG NUMBER {config_number}: learning rate:{lr}, n_steps: {n_steps}, gamma: {gamma}, batch_size: {batch_size}, ent_coef: {ent_coef}")
                        log_file.write(f"\nCONFIG NUMBER {config_number}: lr={lr}, n_steps={n_steps}, gamma={gamma}, batch_size={batch_size}, ent_coef={ent_coef}\n")

                        
                        # Create environment (use training sequences)
                        train_env = TrainPPORecommendationEnv(
                            train_sequences_idx=train_sequences_idx,
                            ratings_dict=ratings_dict,
                            user_embeddings=user_embeddings,
                            movie_embeddings=movie_embeddings,
                            n_history=5,
                            T_max=10
                        )
                        # Wrap environment in a vectorized wrapper
                        vec_env = DummyVecEnv([lambda: train_env])
                        
                        model = PPO(
                            policy='MlpPolicy',
                            env=vec_env,
                            learning_rate=lr,
                            n_steps=n_steps,
                            batch_size=batch_size,
                            gamma=gamma,
                            ent_coef=ent_coef,
    
                            n_epochs=10,
                            gae_lambda=0.95,
                            clip_range=0.2,
                            vf_coef=0.5,
                            max_grad_norm=0.5,
    
                            tensorboard_log=f"../results/{model_name}_config_{config_number}",
                            verbose=0,
                            seed=seed,
                            # device=device
                            device='cpu'
                        )
    
                        model.learn(total_timesteps=total_timesteps)
                        
                        # After training, evaluate on validation (or get best from callback)
                        # For simplicity, we evaluate again with the final model
                        eval_env = EvalPPORecommendationEnv(
                            train_sequences_idx=train_sequences_idx,
                            test_sequences_idx=valid_sequences_idx,
                            ratings_dict=ratings_dict,
                            user_embeddings=user_embeddings,
                            movie_embeddings=movie_embeddings,
                            evaluable_users=evaluable_users,
                            n_history=5,
                            T_max=10
                        )
                        results = evaluate_ppo(model, eval_env, evaluable_users, top_k_list=[5, 10], max_steps=10)
                        for metric, value in results.items():
                            print(f"{metric}: {value:.4f}")
    
                        # One method decides model's quality: precision@5, precision@10, NDCG@5, NDCG@10
                        val_ndcg = results['NDCG@10']
                        if val_ndcg > best_ndcg:
                            best_ndcg = val_ndcg
                            best_config = {
                                'learning_rate': lr,
                                'n_steps': n_steps,
                                'batch_size': batch_size,
                                'gamma': gamma,
                                'ent_coef': ent_coef,
                                'Precision@5': results['Precision@5'],
                                'Precision@10': results['Precision@10'],
                                'NDCG@5': results['NDCG@5'],
                                'NDCG@10': results['NDCG@10']
                            }
                            print(f"New best NDCG@10: {val_ndcg:.4f} with {best_config}")
                            log_file.write(f"New best NDCG@10: {val_ndcg:.4f} with {best_config}\n")

                            model.save(f"../models/best_ppo_CONFIG_{config_number}")
                        config_number += 1
    
    end_time = time.time()
    print("Training done")
    
    # Calculate run time
    training_duration = end_time - start_time
    time_in_minutes_and_seconds = convert(training_duration)
    
    log_file.write(f"Best config: {best_config} with NDCG@10={best_ndcg:.4f}\n")
    log_file.write(f"Total time: {time_in_minutes_and_seconds}\n")


Training started

CONFIG NUMBER 0: learning rate:0.0001, n_steps: 1024, gamma: 0.95, batch_size: 64, ent_coef: 0.0
Precision@5: 0.6656
NDCG@5: 0.6884
Precision@10: 0.6490
NDCG@10: 0.7052
New best NDCG@10: 0.7052 with {'learning_rate': 0.0001, 'n_steps': 1024, 'batch_size': 64, 'gamma': 0.95, 'ent_coef': 0.0, 'Precision@5': 0.6655619162398313, 'Precision@10': 0.6490207893943959, 'NDCG@5': 0.6883816770708258, 'NDCG@10': 0.7051992955383938}

CONFIG NUMBER 1: learning rate:0.0001, n_steps: 1024, gamma: 0.95, batch_size: 64, ent_coef: 0.01
Precision@5: 0.6694
NDCG@5: 0.6905
Precision@10: 0.6498
NDCG@10: 0.7061
New best NDCG@10: 0.7061 with {'learning_rate': 0.0001, 'n_steps': 1024, 'batch_size': 64, 'gamma': 0.95, 'ent_coef': 0.01, 'Precision@5': 0.6693582404338657, 'Precision@10': 0.6498342874359747, 'NDCG@5': 0.6905052436325528, 'NDCG@10': 0.706084323938342}

CONFIG NUMBER 2: learning rate:0.0001, n_steps: 1024, gamma: 0.95, batch_size: 64, ent_coef: 0.05
Precision@5: 0.6791
NDCG@5: 0.698

In [13]:
print(f"Time taken: {time_in_minutes_and_seconds}")

# best_config = { 'learning_rate': lr, 'n_steps': n_steps, 'batch_size': batch_size, 'gamma': gamma, 'ent_coef': ent_coef }
print(f"Best config: {best_config} with NDCG@10={best_ndcg:.4f}\n")

Time taken: 10:05:59
Best config: {'learning_rate': 0.0001, 'n_steps': 1024, 'batch_size': 128, 'gamma': 0.95, 'ent_coef': 0.01, 'Precision@5': 0.6920759264838807, 'Precision@10': 0.6697499246761072, 'NDCG@5': 0.7118415965492906, 'NDCG@10': 0.7258625313078175} with NDCG@10=0.7259



In [ ]:
# Time taken: 10:05:59

# Best config: {'learning_rate': 0.0001, 'n_steps': 1024, 'batch_size': 128, 'gamma': 0.95, 'ent_coef': 0.01, 'Precision@5': 0.6920759264838807, 'Precision@10': 0.6697499246761072, 'NDCG@5': 0.7118415965492906, 'NDCG@10': 0.7258625313078175} 